# Microbenchmarks on CPU
This is a notebook for microbenchmarks running on CPU.

In [ ]:
# Bootstrap defaults similar to GPU notebook
import os, glob, psutil

# Ensure SPARK_HOME is set (fallback to local build dist)
SPARK_HOME = os.environ.get("SPARK_HOME", os.environ["HOME"] + "/spark/dist")
os.environ["SPARK_HOME"] = SPARK_HOME
os.environ['JAVA_HOME'] = os.environ.get('JAVA_HOME', '/usr/lib/jvm/java-17-openjdk-amd64')

# Clean up lingering SparkSubmit JVMs that can cause empty Py4J answers
for p in psutil.process_iter(['pid','name','cmdline']):
    cl = ' '.join(p.info.get('cmdline') or [])
    if p.info.get('name') == 'java' and 'org.apache.spark.deploy.SparkSubmit' in cl:
        try:
            p.kill()
        except Exception:
            pass

# Ensure findspark can locate Spark
try:
    import findspark; findspark.init(os.environ['SPARK_HOME'])
except Exception:
    pass

# Use local mode unless a cluster URL is provided
os.environ.setdefault("SPARK_MASTER_URL", "local[*]")

# Data and event log defaults
os.environ.setdefault("DATA_ROOT", os.environ["HOME"] + "/spark-rapids-examples/datasets")
os.environ.setdefault("EVENTLOG_DIR", "/tmp/spark-events")
try:
    os.makedirs(os.environ["EVENTLOG_DIR"], exist_ok=True)
except Exception:
    pass

print("SPARK_HOME =", os.environ["SPARK_HOME"]) 
print("SPARK_MASTER_URL =", os.environ["SPARK_MASTER_URL"]) 
print("EVENTLOG_DIR =", os.environ["EVENTLOG_DIR"]) 


Run the microbenchmark with retry times

In [ ]:
def runMicroBenchmark(spark, appName, query, retryTimes):
    count = 0
    total_time = 0
    # You can print the physical plan of each query
    # spark.sql(query).explain()
    while count < retryTimes:
        start = time.time()
        spark.sql(query).collect()
        end = time.time()
        total_time += round(end - start, 2)
        count = count + 1
        print("Retry times : {}, ".format(count) + appName + " microbenchmark takes {} seconds".format(round(end - start, 2)))
    print(appName + " microbenchmark takes average {} seconds after {} retries".format(round(total_time/retryTimes),retryTimes))
    with open('result.txt', 'a') as file:
        file.write("{},{},{}\n".format(appName, round(total_time/retryTimes), retryTimes))

In [ ]:
# You need to update data path with your real path and hardware resource!
from pyspark.sql import SparkSession
from pyspark.conf import SparkConf
import time, os

# Java 17 add-opens flags (harmless on Java 8+)
_DEF_OPENS = (
    "--add-opens=java.base/java.lang=ALL-UNNAMED "
    "--add-opens=java.base/java.lang.invoke=ALL-UNNAMED "
    "--add-opens=java.base/java.lang.reflect=ALL-UNNAMED "
    "--add-opens=java.base/java.io=ALL-UNNAMED "
    "--add-opens=java.base/java.net=ALL-UNNAMED "
    "--add-opens=java.base/java.nio=ALL-UNNAMED "
    "--add-opens=java.base/java.util=ALL-UNNAMED "
    "--add-opens=java.base/java.util.concurrent=ALL-UNNAMED "
    "--add-opens=java.base/java.util.concurrent.atomic=ALL-UNNAMED "
    "--add-opens=java.base/jdk.internal.ref=ALL-UNNAMED "
    "--add-opens=java.base/sun.nio.ch=ALL-UNNAMED "
    "--add-opens=java.base/sun.nio.cs=ALL-UNNAMED "
    "--add-opens=java.base/sun.security.action=ALL-UNNAMED "
    "--add-opens=java.base/sun.util.calendar=ALL-UNNAMED"
)

# Build base conf (GPU-style settings but CPU-only)
base = (SparkConf()
    .setMaster(os.environ.get("SPARK_MASTER_URL", "local[*]"))
    .setAppName("Microbenchmark on CPU")
    .set("spark.driver.memory", os.environ.get("DRIVER_MEM", "12g"))
    .set("spark.sql.adaptive.enabled", "true")
    .set("spark.sql.files.maxPartitionBytes", os.environ.get("MAX_PARTITION_BYTES", "128m"))
    .set("spark.sql.shuffle.partitions", os.environ.get("SHUFFLE_PARTITIONS", "32"))
    .set("spark.locality.wait", "0")
    .set("spark.scheduler.mode", "FAIR")
    .set("spark.eventLog.enabled", "false")
    .set("spark.driver.extraJavaOptions", _DEF_OPENS)
    .set("spark.executor.extraJavaOptions", _DEF_OPENS)
)

# create spark session
spark = SparkSession.builder.config(conf=base).getOrCreate()


In [ ]:
dataRoot = os.environ["HOME"] + "/spark-rapids-examples/datasets"

# Load dataframe and create tempView (avoid extreme repartition)
spark.read.parquet(dataRoot + "/tpcds/customer").createOrReplaceTempView("customer")
spark.read.parquet(dataRoot + "/tpcds/store_sales").createOrReplaceTempView("store_sales")
spark.read.parquet(dataRoot + "/tpcds/catalog_sales").createOrReplaceTempView("catalog_sales")
spark.read.parquet(dataRoot + "/tpcds/web_sales").createOrReplaceTempView("web_sales")
spark.read.parquet(dataRoot + "/tpcds/item").createOrReplaceTempView("item")
spark.read.parquet(dataRoot + "/tpcds/date_dim").createOrReplaceTempView("date_dim")

# Cache hot tables to reduce I/O contention during parallel runs
for t in ("customer","store_sales","catalog_sales","web_sales","item","date_dim"):
    try:
        spark.catalog.cacheTable(t)
    except Exception:
        pass
# Materialize caches once to warm up
for t in ("customer","store_sales","catalog_sales","web_sales","item","date_dim"):
    _ = spark.table(t).count()

print("-"*50)
time.sleep(2)

### Expand&HashAggregate
This is a microbenchmark about Expand&HashAggregate expressions running on the CPU. The query calculates the distinct value of some dimension columns and average birth year by different c_salutation of customers after grouping by c_current_hdemo_sk.

In [ ]:
query0 = '''
select c_current_hdemo_sk,
count(DISTINCT if(c_salutation=="Ms.",c_salutation,null)) as c1,
count(DISTINCT if(c_salutation=="Mr.",c_salutation,null)) as c12,
count(DISTINCT if(c_salutation=="Dr.",c_salutation,null)) as c13,

count(DISTINCT if(c_salutation=="Ms.",c_first_name,null)) as c2,
count(DISTINCT if(c_salutation=="Mr.",c_first_name,null)) as c22,
count(DISTINCT if(c_salutation=="Dr.",c_first_name,null)) as c23,

count(DISTINCT if(c_salutation=="Ms.",c_last_name,null)) as c3,
count(DISTINCT if(c_salutation=="Mr.",c_last_name,null)) as c32,
count(DISTINCT if(c_salutation=="Dr.",c_last_name,null)) as c33,

count(DISTINCT if(c_salutation=="Ms.",c_birth_country,null)) as c4,
count(DISTINCT if(c_salutation=="Mr.",c_birth_country,null)) as c42,
count(DISTINCT if(c_salutation=="Dr.",c_birth_country,null)) as c43,

count(DISTINCT if(c_salutation=="Ms.",c_email_address,null)) as c5,
count(DISTINCT if(c_salutation=="Mr.",c_email_address,null)) as c52,
count(DISTINCT if(c_salutation=="Dr.",c_email_address,null)) as c53,

count(DISTINCT if(c_salutation=="Ms.",c_login,null)) as c6,
count(DISTINCT if(c_salutation=="Mr.",c_login,null)) as c62,
count(DISTINCT if(c_salutation=="Dr.",c_login,null)) as c63,

count(DISTINCT if(c_salutation=="Ms.",c_preferred_cust_flag,null)) as c7,
count(DISTINCT if(c_salutation=="Mr.",c_preferred_cust_flag,null)) as c72,
count(DISTINCT if(c_salutation=="Dr.",c_preferred_cust_flag,null)) as c73,

count(DISTINCT if(c_salutation=="Ms.",c_birth_month,null)) as c8,
count(DISTINCT if(c_salutation=="Mr.",c_birth_month,null)) as c82,
count(DISTINCT if(c_salutation=="Dr.",c_birth_month,null)) as c83,

avg(if(c_salutation=="Ms.",c_birth_year,null)) as avg1,
avg(if(c_salutation=="Mr.",c_birth_year,null)) as avg2,
avg(if(c_salutation=="Dr.",c_birth_year,null)) as avg3,
avg(if(c_salutation=="Miss.",c_birth_year,null)) as avg4,
avg(if(c_salutation=="Mrs.",c_birth_year,null)) as avg5,
avg(if(c_salutation=="Sir.",c_birth_year,null)) as avg6,
avg(if(c_salutation=="Professor.",c_birth_year,null)) as avg7,
avg(if(c_salutation=="Teacher.",c_birth_year,null)) as avg8,
avg(if(c_salutation=="Agent.",c_birth_year,null)) as avg9,
avg(if(c_salutation=="Director.",c_birth_year,null)) as avg10
from customer group by c_current_hdemo_sk
'''
print("-"*50)

In [ ]:
# Run microbenchmark with n retry time
runMicroBenchmark(spark,"Expand&HashAggregate", query0, 2)
time.sleep(2)

### Windowing (without data skew)
This is a microbenchmark about windowing expressions running on CPU mode. The sub-query calculates the average ss_sales_price of a fixed window function partition by ss_customer_sk, and the parent query calculates the average price of the sub-query grouping by each customer.

In [ ]:
query1 = '''
select ss_customer_sk,avg(avg_price) as avg_price
from
(
SELECT ss_customer_sk ,avg(ss_sales_price) OVER (PARTITION BY ss_customer_sk order by ss_sold_date_sk ROWS BETWEEN 50 PRECEDING AND 50 FOLLOWING ) as avg_price
FROM store_sales
where ss_customer_sk is not null
) group by ss_customer_sk order by 2 desc 
'''
print("-"*50)

In [ ]:
# Run microbenchmark with n retry time
runMicroBenchmark(spark,"Windowing without skew", query1, 2)
time.sleep(2)

### Windowing(with data skew)
Data skew is caused by many null values in the ss_customer_sk column.

In [ ]:
query2 = '''
select ss_customer_sk,avg(avg_price) as avg_price
from
(
SELECT ss_customer_sk ,avg(ss_sales_price) OVER (PARTITION BY ss_customer_sk order by ss_sold_date_sk ROWS BETWEEN 50 PRECEDING AND 50 FOLLOWING ) as avg_price
FROM store_sales
) group by ss_customer_sk order by 2 desc 
'''
print("-"*50)

In [ ]:
# Run microbenchmark with n retry time
runMicroBenchmark(spark,"Windowing with skew", query2, 2)
time.sleep(2)

### Intersection
This is a microbenchmark about intersection operation running on CPU mode. The query calculates items in the same brand, class, and category that are sold in all three sales channels in two consecutive years.

In [ ]:
query3 = '''
select i_item_sk ss_item_sk
 from item,
    (select iss.i_brand_id brand_id, iss.i_class_id class_id, iss.i_category_id category_id
     from store_sales, item iss, date_dim d1
     where ss_item_sk = iss.i_item_sk
                    and ss_sold_date_sk = d1.d_date_sk
       and d1.d_year between 1999 AND 1999 + 2
   intersect
     select ics.i_brand_id, ics.i_class_id, ics.i_category_id
     from catalog_sales, item ics, date_dim d2
     where cs_item_sk = ics.i_item_sk
       and cs_sold_date_sk = d2.d_date_sk
       and d2.d_year between 1999 AND 1999 + 2
   intersect
     select iws.i_brand_id, iws.i_class_id, iws.i_category_id
     from web_sales, item iws, date_dim d3
     where ws_item_sk = iws.i_item_sk
       and ws_sold_date_sk = d3.d_date_sk
       and d3.d_year between 1999 AND 1999 + 2) x
 where i_brand_id = brand_id
   and i_class_id = class_id
   and i_category_id = category_id
'''

In [ ]:
# Run microbenchmark with n retry time
runMicroBenchmark(spark,"NDS Q14a subquery", query3, 2)
time.sleep(2)

In [ ]:
# Run the 4 micro-benchmarks concurrently on one SparkSession/CPU
# Requires: query0, query1, query2, query3 already defined; temp views already created.

from concurrent.futures import ThreadPoolExecutor, as_completed
import time, os

# Tune quickly if you hit contention
spark.conf.set("spark.sql.files.maxPartitionBytes", os.environ.get("MB_MAX_PART_BYTES", "128m"))

def run_queries_in_pool(pool_name: str, query, retryTimes: int = 1):
    count = 0
    sc = spark.sparkContext
    t0 = time.time()
    sc.setLocalProperty("spark.scheduler.pool", pool_name)  # assign this job to a pool
    sc.setJobGroup(f"{pool_name}", f"{pool_name} run", True)
    while count < retryTimes:
        print(f"run {count+1} of {retryTimes} for {pool_name}")
        spark.sql(query).collect()  # blocking action submits a job
        count += 1
    return pool_name, round(time.time() - t0, 1)

jobs = [
    ("poolA", query0),
    ("poolB", query1),
    ("poolC", query2),
    ("poolD", query3),
    ("poolE", query0),
    ("poolF", query1),
    ("poolG", query2),
    ("poolH", query3),
    
]

# You can tune the parallelism here quickly if you hit contention
PARALLEL_JOBS = int(os.environ.get("MB_PARALLEL_JOBS", "8"))

with ThreadPoolExecutor(max_workers=PARALLEL_JOBS) as ex:
    futs = [ex.submit(run_queries_in_pool, p, query) for p, query in jobs[:PARALLEL_JOBS]]
    for f in as_completed(futs):
        name, secs = f.result()
        print(f"{name} took {secs}s")

